In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


In [19]:
import urllib.request

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/water-treatment/water-treatment.data"
local_path = "water-treatment.data"

urllib.request.urlretrieve(url, local_path)
print("Downloaded:", local_path)

# The file is typically comma-separated with '?' missing values.
df = pd.read_csv(local_path, header=None, na_values='?')
df.head()


Downloaded: water-treatment.data


,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,D-1/3/90,44101.0,1.5,7.8,NaN,407.0,166.0,66.3,4.5,2110,...,2000.0,NaN,58.8,95.5,NaN,70.0,NaN,79.4,87.3,99.6
1,D-2/3/90,39024.0,3.0,7.7,NaN,443.0,214.0,69.2,6.5,2660,...,2590.0,NaN,60.7,94.8,NaN,80.8,NaN,79.5,92.1,100.0
2,D-4/3/90,32229.0,5.0,7.6,NaN,528.0,186.0,69.9,3.4,1666,...,1888.0,NaN,58.2,95.6,NaN,52.9,NaN,75.8,88.7,98.5
3,D-5/3/90,35023.0,3.5,7.9,205.0,588.0,192.0,65.6,4.5,2430,...,1840.0,33.1,64.2,95.3,87.3,72.3,90.2,82.3,89.6,100.0
4,D-6/3/90,36924.0,1.5,8.0,242.0,496.0,176.0,64.8,4.0,2110,...,2120.0,NaN,62.7,95.6,NaN,71.0,92.1,78.2,87.5,99.5


In [23]:
cols = [
    "DATE",
    "Q-E","ZN-E","PH-E","DBO-E","DQO-E","SS-E","SSV-E","SED-E","COND-E",
    "PH-P","DBO-P","DQO-P","SS-P","SSV-P","SED-P","COND-P",
    "PH-D","DBO-D","DQO-D","SS-D","SSV-D","SED-D","COND-D",
    "PH-S","DBO-S","DQO-S","SS-S","SSV-S","SED-S","COND-S",
    "RD-DBO-P","RD-DQO-P","RD-SS-P","RD-SED-P",
    "RD-DBO-S","RD-DQO-S","RD-SS-S","RD-SED-S"
]

print("Column count:", len(cols))
print("DataFrame columns:", df.shape[1])

df.columns = cols
df.head()


Column count: 39
DataFrame columns: 39


,DATE,Q-E,ZN-E,PH-E,DBO-E,DQO-E,SS-E,SSV-E,SED-E,COND-E,...,SED-S,COND-S,RD-DBO-P,RD-DQO-P,RD-SS-P,RD-SED-P,RD-DBO-S,RD-DQO-S,RD-SS-S,RD-SED-S
0,D-1/3/90,44101.0,1.5,7.8,NaN,407.0,166.0,66.3,4.5,2110,...,2000.0,NaN,58.8,95.5,NaN,70.0,NaN,79.4,87.3,99.6
1,D-2/3/90,39024.0,3.0,7.7,NaN,443.0,214.0,69.2,6.5,2660,...,2590.0,NaN,60.7,94.8,NaN,80.8,NaN,79.5,92.1,100.0
2,D-4/3/90,32229.0,5.0,7.6,NaN,528.0,186.0,69.9,3.4,1666,...,1888.0,NaN,58.2,95.6,NaN,52.9,NaN,75.8,88.7,98.5
3,D-5/3/90,35023.0,3.5,7.9,205.0,588.0,192.0,65.6,4.5,2430,...,1840.0,33.1,64.2,95.3,87.3,72.3,90.2,82.3,89.6,100.0
4,D-6/3/90,36924.0,1.5,8.0,242.0,496.0,176.0,64.8,4.0,2110,...,2120.0,NaN,62.7,95.6,NaN,71.0,92.1,78.2,87.5,99.5


In [25]:
numeric_cols = [c for c in df.columns if c != "DATE"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")


In [26]:
corr_ph = df[["PH-E","PH-S"]].corr().iloc[0,1]
print("Correlation(PH-E, PH-S) =", corr_ph)


Correlation(PH-E, PH-S) = -0.032146805841816485


In [29]:
df["SAFE-PH-S"] = np.where((df["PH-S"] > 6.5) & (df["PH-S"] <= 8.5), "yes", "no")
print(df["SAFE-PH-S"].value_counts())


SAFE-PH-S
no     512
yes     15
Name: count, dtype: int64


In [30]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.tree import DecisionTreeClassifier

inputs = ["Q-E","ZN-E","PH-E","DBO-E","DQO-E","SS-E","SSV-E","SED-E","COND-E"]
X = df[inputs]
y = df["SAFE-PH-S"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

dt = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", DecisionTreeClassifier(random_state=42))
])

# 10-fold CV
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_res = cross_validate(dt, X_train, y_train, cv=cv,
                        scoring=["accuracy","precision","recall","f1"])

print("Decision Tree — 10-fold CV (Train 80%)")
print("Accuracy :", cv_res["test_accuracy"].mean())
print("Precision:", cv_res["test_precision"].mean())
print("Recall   :", cv_res["test_recall"].mean())
print("F1       :", cv_res["test_f1"].mean())

# Test evaluation (20%)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)

print("\nDecision Tree — TEST (20%)")
print("Accuracy:", accuracy_score(y_test, pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_dt, labels=["yes","no"]))
print(classification_report(y_test, pred_dt, digits=4))

proba_dt = dt.predict_proba(X_test)
yes_idx = list(dt.named_steps["clf"].classes_).index("yes")
print("ROC-AUC:", roc_auc_score((y_test=="yes").astype(int), proba_dt[:, yes_idx]))


Decision Tree — 10-fold CV (Train 80%)
Accuracy : 0.9336655592469546
Precision: nan
Recall   : nan
F1       : nan

Decision Tree — TEST (20%)
Accuracy: 0.9433962264150944
Confusion Matrix:
 [[  0   3]
 [  3 100]]
              precision    recall  f1-score   support

          no     0.9709    0.9709    0.9709       103
         yes     0.0000    0.0000    0.0000         3

    accuracy                         0.9434       106
   macro avg     0.4854    0.4854    0.4854       106
weighted avg     0.9434    0.9434    0.9434       106

ROC-AUC: 0.4854368932038835


In [31]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42))
])

# 10-fold CV
cv_res = cross_validate(lr, X_train, y_train, cv=cv,
                        scoring=["accuracy","precision","recall","f1"])

print("Logistic Regression — 10-fold CV (Train 80%)")
print("Accuracy :", cv_res["test_accuracy"].mean())
print("Precision:", cv_res["test_precision"].mean())
print("Recall   :", cv_res["test_recall"].mean())
print("F1       :", cv_res["test_f1"].mean())

# Test evaluation (20%)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("\nLogistic Regression — TEST (20%)")
print("Accuracy:", accuracy_score(y_test, pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_lr, labels=["yes","no"]))
print(classification_report(y_test, pred_lr, digits=4))

proba_lr = lr.predict_proba(X_test)
yes_idx = list(lr.named_steps["clf"].classes_).index("yes")
print("ROC-AUC:", roc_auc_score((y_test=="yes").astype(int), proba_lr[:, yes_idx]))


Logistic Regression — 10-fold CV (Train 80%)
Accuracy : 0.9667774086378736
Precision: nan
Recall   : nan
F1       : nan

Logistic Regression — TEST (20%)
Accuracy: 0.9716981132075472
Confusion Matrix:
 [[  0   3]
 [  0 103]]
              precision    recall  f1-score   support

          no     0.9717    1.0000    0.9856       103
         yes     0.0000    0.0000    0.0000         3

    accuracy                         0.9717       106
   macro avg     0.4858    0.5000    0.4928       106
weighted avg     0.9442    0.9717    0.9578       106

ROC-AUC: 0.5372168284789645
